# 12 因果推論與政策評估

淋浴暴露真的「導致」感染嗎？消毒水系統真的有效嗎？

流程：**DAG 因果圖 → 干擾/中介/碰撞 → 歸因風險 AR/PAR → DiD 介入評估 → 平行趨勢檢驗**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: DAG（有向無環圖）---
# 用文字描述因果關係（不需要安裝 graphviz）
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# DAG 視覺化
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

# 節點
nodes = {
    "floor/wing": (1, 5.5),
    "water\ncontamination": (3.5, 5.5),
    "shower\naerosol": (6, 5.5),
    "infection": (8.5, 5.5),
    "functional\nstatus": (1, 3),
    "shower\nuse": (4, 3),
    "age": (1, 1),
    "comorbidities": (3.5, 1),
    "severity": (6, 1),
    "death": (8.5, 1),
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Rectangle((x-0.7, y-0.4), 1.4, 0.8,
                 fill=True, facecolor="#e0e0e0", edgecolor="black", linewidth=1.5))
    ax.text(x, y, name, ha="center", va="center", fontsize=8, fontweight="bold")

# 箭頭（因果方向）
arrows = [
    ("floor/wing", "water\ncontamination"),
    ("water\ncontamination", "shower\naerosol"),
    ("shower\naerosol", "infection"),
    ("functional\nstatus", "shower\nuse"),
    ("shower\nuse", "infection"),
    ("functional\nstatus", "infection"),
    ("age", "comorbidities"),
    ("comorbidities", "severity"),
    ("severity", "death"),
    ("infection", "severity"),
]

for src, dst in arrows:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.annotate("", xy=(x2-0.7, y2), xytext=(x1+0.7, y1),
                arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

ax.set_title("Legionella DAG \u2014 因果關係圖", fontsize=14)
plt.tight_layout()
plt.show()

print("=== 因果結構辨識 ===")
print("干擾因子：functional_status \u2192 shower_use 且 \u2192 infection")
print("中介變項：shower_aerosol 在 water_contamination \u2192 infection 之間")
print("碰撞因子：hospitalized \u2190 severity 且 \u2190 infection")
print("\n\u2192 控制干擾因子（Ch05 已做）= 正確")
print("\u2192 控制碰撞因子 = 錯誤！會產生假性關聯")

In [ ]:
# --- Step 2: 歸因風險（Attributable Risk）---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 淋浴暴露的侵襲率
exposed = df[df["shower_use"] == 1]
unexposed = df[df["shower_use"] == 0]

risk_exposed = exposed["infected"].mean()
risk_unexposed = unexposed["infected"].mean()
risk_total = df["infected"].mean()

print("=== 淋浴暴露的侵襲率 ===")
print(f"淋浴者侵襲率：{risk_exposed:.1%} ({exposed['infected'].sum()}/{len(exposed)})")
print(f"非淋浴者侵襲率：{risk_unexposed:.1%} ({unexposed['infected'].sum()}/{len(unexposed)})")
print(f"全體侵襲率：{risk_total:.1%}")

# Attributable Risk
AR = risk_exposed - risk_unexposed
print(f"\n=== Attributable Risk (AR) ===")
print(f"AR = {risk_exposed:.3f} - {risk_unexposed:.3f} = {AR:.3f}")
print(f"\u2192 淋浴者比非淋浴者多 {AR:.1%} 的感染風險")

# Population Attributable Risk
PAR = risk_total - risk_unexposed
PAR_pct = PAR / risk_total * 100
print(f"\n=== Population Attributable Risk (PAR) ===")
print(f"PAR = {risk_total:.3f} - {risk_unexposed:.3f} = {PAR:.3f}")
print(f"PAR% = {PAR_pct:.1f}%")
print(f"\u2192 如果消除淋浴暴露，理論上可減少 {PAR_pct:.0f}% 的感染")
print("\u2192 前提：因果關係成立，且無其他傳播途徑")

In [ ]:
# --- Step 3: 反事實思考 ---
n_total = len(df)
n_infected = df["infected"].sum()

# 反事實：如果所有人都不淋浴
counterfactual_cases = int(n_total * risk_unexposed)
prevented = n_infected - counterfactual_cases

print("=== 反事實情境 ===")
print(f"實際感染人數：{n_infected}")
print(f"如果所有人都不淋浴（反事實）：預期 {counterfactual_cases} 人感染")
print(f"可預防的感染數：{prevented}")
print(f"\n\u2192 但這只是理論估算！")
print("\u2192 實際上，禁止所有人淋浴不可行")
print("\u2192 更實際的做法：消毒水系統，讓淋浴變安全")

In [ ]:
# --- Step 4: DiD 資料準備 ---
# 情境：1月25日對 2-3F B翼 實施水系統緊急消毒
# 介入組：2-3F B翼（高侵襲率，靠近汙染源）
# 對照組：1F 全部 + 2-3F A翼（不同水源或非目標區）

df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
cases = df[df["infected"] == 1].copy()

# 建立每日面板資料
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# 介入組：2-3F B翼
treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]
treated_daily = treated_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 對照組：其餘區域
control_cases = cases[~((cases["floor"].isin([2, 3])) & (cases["wing"] == "B"))]
control_daily = control_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 組合成面板
panel = pd.DataFrame({
    "date": list(all_dates) * 2,
    "treated": [1] * len(all_dates) + [0] * len(all_dates),
    "daily_cases": list(treated_daily.values) + list(control_daily.values),
})
panel["post"] = (panel["date"] >= "2026-01-25").astype(int)
panel["day"] = (panel["date"] - panel["date"].min()).dt.days

print("=== DiD 面板資料 ===")
print(f"介入組（2-3F B翼）：{len(treated_daily)} 天")
print(f"對照組（其餘區域）：{len(control_daily)} 天")
print(f"介入日：2026-01-25")
print(f"\n介入前後病例數：")
summary = panel.groupby(["treated", "post"]).agg(total=('daily_cases','sum'), mean=('daily_cases','mean')).reset_index()
summary["group"] = summary["treated"].map({1: "介入組(2-3F B)", 0: "對照組"})
summary["period"] = summary["post"].map({0: "介入前", 1: "介入後"})
print(summary[["group", "period", "total", "mean"]].to_string(index=False))

In [ ]:
# --- Step 5: 平行趨勢檢驗 + DiD 視覺化 ---
fig, ax = plt.subplots(figsize=(10, 5))

# 介入組
ax.plot(all_dates, treated_daily.values, marker="o", markersize=4,
        label="介入組 (2-3F B翼)", color="#e34a33")
# 對照組
ax.plot(all_dates, control_daily.values, marker="s", markersize=4,
        label="對照組 (其餘)", color="#2c7fb8")

# 介入線
ax.axvline(x=pd.Timestamp("2026-01-25"), color="black", linestyle="--",
           alpha=0.7, label="介入日 (1/25)")

ax.set_title("DiD \u2014 介入前後病例數趨勢")
ax.set_xlabel("日期")
ax.set_ylabel("每日病例數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\u2192 觀察介入前（虛線左側）兩條線是否大致平行")
print("\u2192 如果平行，DiD 估計較可信")

In [ ]:
# --- Step 6: DiD OLS 迴歸 ---
model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit()

print("=== DiD 迴歸結果 ===")
print(model.summary().tables[1])

did_effect = model.params["treated:post"]
did_p = model.pvalues["treated:post"]

print(f"\n=== DiD 效果估計 ===")
print(f"treated:post 係數 = {did_effect:.3f}")
print(f"p-value = {did_p:.4f}")

if did_effect < 0:
    print(f"\n\u2192 介入後，介入組每日病例數比預期減少 {abs(did_effect):.1f} 人")
else:
    print(f"\n\u2192 介入後，介入組每日病例數比預期增加 {did_effect:.1f} 人")

if did_p < 0.05:
    print("\u2192 效果統計顯著（p < 0.05）")
else:
    print("\u2192 效果未達統計顯著（p \u2265 0.05）")
    print("\u2192 可能原因：樣本量不足、觀察期太短、介入效果需要更長時間顯現")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| DAG | 用圖形辨識干擾、中介、碰撞因子 |
| AR / PAR | 量化暴露對感染的貢獻比例 |
| 反事實 | 估算「如果消除暴露」的預期效果 |
| DiD | `cases ~ treated + post + treated:post` |
| 平行趨勢 | 介入前兩組趨勢是否一致 |

**結論**：
- DAG 幫我們釐清哪些變項該控制、哪些不該控制
- AR/PAR 量化暴露的貢獻，但前提是因果關係成立
- DiD 是評估介入效果的準實驗方法，但需要平行趨勢假設
- 在觀察性資料中，因果推論永遠需要謹慎

下一章（Ch13），我們確保所有分析可重現 → 可重現研究。